In [7]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
import rioxarray as rioxr
from shapely import box

import pystac_client
import planetary_computer

from IPython.display import Image


sys.path.append("../utils")



In [8]:
# Reading in data, dropping all columns except what we need for this function: inspection_id and geometry
inspections = gpd.read_file(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master.geojson"
)[["inspection_id", "geometry"]]


In [9]:
inspections.shape

(67580, 2)

In [10]:
def calculate_landcover(inspections, id_col="inspection_id"):
    
    """
    This function is used to calculate the majority landcover type for every polygon geometry in the dataset. 
    This will only work in Santa Barbara County, CA; any geometries outside the bounding box will return NaN. 
    Landcover statistics are sourced from Microsoft Planetary Computer. 
    
    The function iterates over every row in the dataset, and relies on the presence of an 'inspection_id' column, 
    which is a unique identifier for every inspection. 
    
    Usage example:
    desired_data = calculate_landcover(input_data)
    
    
    """
    
    from rioxarray.exceptions import NoDataInBounds, OneDimensionalRaster
    
    inspections = inspections[['inspection_id', 'geometry']]
    
    # Bounding box surrounding SB County
    sb_bbox = [
        -120.70595549829301,
        34.26000089800273,
        -119.00465428757012,
        35.126360340575715,
    ]

    # Read landcover data from Microsoft Planetary Computer
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    # Search for landcover item, and save item
    search = catalog.search(collections=["gap"], bbox=sb_bbox)
    items = search.item_collection()
    print(f"Returned {len(items)} Items from Microsoft Planetary Computer")
    item = items[0]

    # Open landcover data with rioxr
    landcover = rioxr.open_rasterio(item.assets["data"].href)
    # Squeeze
    landcover = landcover.squeeze().drop_vars("band")

    # Convert sb_bbox to a polygon
    sb_bbox = box(*sb_bbox)
    sb_bbox = gpd.GeoDataFrame({"geometry": [sb_bbox]}, crs="EPSG:4326")
    sb_bbox = sb_bbox.to_crs(landcover.rio.crs)
    assert sb_bbox.crs == landcover.rio.crs

    landcover = landcover.rio.write_nodata(0)

    # Clip landcover data to sb_bbox
    landcover_clip = landcover.rio.clip_box(*sb_bbox.total_bounds).rio.clip(
        sb_bbox.geometry
    )

    # Now working with inspections data, checking CRS
    inspections = inspections.to_crs(landcover.rio.crs)
    assert inspections.crs == landcover.rio.crs

    # Now looping over every row, calculating landcover by individual geometry
    # _________________________________________________
    
    maj_codes = []
    
    for _, row in inspections.iterrows():
        # addr = row[id_col]
        geom = row.geometry
        minx, miny, maxx, maxy = geom.bounds
        
        try:
            clipped = landcover.rio.clip_box(minx, miny, maxx, maxy).rio.clip([geom])

        except NoDataInBounds:
            maj_codes.append(np.nan)
            continue

        except OneDimensionalRaster:
            # tiny slice: allow one‐dim on the bbox step, skip the geometry mask
            clipped = landcover.rio.clip_box(
                minx, miny, maxx, maxy, allow_one_dimensional_raster=True
            )

        # count codes
        vals, counts = np.unique(clipped.values, return_counts=True)
        pix_counts = (
            pd.DataFrame({"code": vals, "count": counts})
            .query("code != 0")
        )

        if pix_counts.empty:
            maj_codes.append(np.nan)
        else:
            maj_codes.append(pix_counts.loc[pix_counts["count"].idxmax(), "code"])
            
    out = inspections.copy()
    out["maj_landcover_code"] = maj_codes
    out = out[["inspection_id", "maj_landcover_code"]]
    
    # NA Check
    na_count = out["maj_landcover_code"].isna().sum()
    if na_count > 0:
        import warnings

        warnings.warn(
            f"{na_count} inspection(s) have missing maj_landcover_code", UserWarning
        )
        print(f"{na_count} inspection(s) with NA majority landcover code")
    else:
        print("Calculated Majority Landcover Classes Successfully")
    
    return out



In [11]:
calculated_landcover = calculate_landcover(inspections)
calculated_landcover

Returned 2 Items from Microsoft Planetary Computer
4723 inspection(s) with NA majority landcover code


/tmp/ipykernel_674382/2294841881.py:93: UserWarning: 4723 inspection(s) have missing maj_landcover_code
  warnings.warn(


,inspection_id,maj_landcover_code
0,1,304.0
1,2,304.0
2,3,303.0
3,4,39.0
4,5,304.0
...,...,...
67575,67576,39.0
67576,67577,583.0
67577,67578,583.0
67578,67579,583.0


In [12]:
calculated_landcover.to_csv(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspection_id_landcover_buffer_geometries.csv",
    index=False,
)


In [13]:
test = pd.read_csv(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspection_id_landcover_buffer_geometries.csv")
test

,inspection_id,maj_landcover_code
0,1,304.0
1,2,304.0
2,3,303.0
3,4,39.0
4,5,304.0
...,...,...
67575,67576,39.0
67576,67577,583.0
67577,67578,583.0
67578,67579,583.0
